# Quick Start: Five Lines to Structured Data

Discover a schema, extract records, and normalize values — all in five lines of Python.

**What you'll learn:** How `Pipeline.quick()` automates the entire discovery → extraction → normalization pipeline.

**Estimated API cost:** ~$0.01

In [ ]:
# pip install "catchfly[openai,export]"
# export OPENAI_API_KEY="sk-..."

## The five lines

In [ ]:
from catchfly import Pipeline
from catchfly.demo import load_samples
from catchfly.discovery import SinglePassDiscovery
from catchfly.extraction import LLMDirectExtraction
from catchfly.normalization import LLMCanonicalization

docs = load_samples("product_reviews")

# suggested_fields + max_fields guide the LLM to discover a clean, focused schema
pipeline = Pipeline(
    discovery=SinglePassDiscovery(
        model="gpt-5.4-mini",
        max_fields=7,
        suggested_fields=["product_name", "brand", "category", "rating", "price", "pros", "cons"],
    ),
    extraction=LLMDirectExtraction(model="gpt-5.4-mini"),
    normalization=LLMCanonicalization(model="gpt-5.4-mini"),
)

results = pipeline.run(
    docs, domain_hint="Electronics product reviews", normalize_fields=["pros"]
)

In [ ]:
df = results.to_dataframe()
df[["product_name", "brand", "category", "rating", "price", "pros", "cons"]].head(10)

That's it. Catchfly discovered a schema from your documents, extracted structured records, and normalized the "pros" field — grouping variants like "great battery" and "long battery life" into a single canonical form.

## What happened under the hood

### 1. Schema discovery

Catchfly sampled your documents and asked the LLM to infer a schema:

In [ ]:
import json

print(json.dumps(results.schema.json_schema, indent=2))

### 2. Extracted records

Each document was sent to the LLM with the discovered schema. The result: typed Pydantic model instances with provenance tracking.

In [ ]:
for record in results.records[:5]:
    print(f"{record.product_name}: pros={record.pros}")

### 3. Normalization

The "pros" field had messy, inconsistent values across reviews. Catchfly grouped them:

In [ ]:
norm = results.normalizations["pros"]
for canonical, members in norm.clusters.items():
    print(f"  {canonical}: {members}")

### 4. Cost report

In [ ]:
print(f"Total cost: ${results.report.total_cost_usd:.4f}")
print(f"Total tokens: {results.report.total_input_tokens + results.report.total_output_tokens:,}")

estimate = pipeline.estimate_cost(docs)
print(estimate)

In [ ]:
pipeline = Pipeline.quick(model="gpt-5.4-mini")
estimate = pipeline.estimate_cost(docs)
print(estimate)

## Next steps

- [02_rare_disease.ipynb](02_rare_disease.ipynb) — Full pipeline with HPO ontology coding
- [03_product_catalog.ipynb](03_product_catalog.ipynb) — Cost control, checkpoints, per-field normalization
- [04_custom_schema.ipynb](04_custom_schema.ipynb) — Skip discovery, use your own Pydantic models
- [05_local_models.ipynb](05_local_models.ipynb) — Run everything locally with Ollama